# Goalz CV Pipeline - Nature Element Classifier

End-to-end binary image classifier: **Is this image a nature element?**

**Datasets required (add via Kaggle → Add Data):**
- `plantnet/plantnet-300k` → positive class (trees, shrubs, flowers, herbs)
- `benjaminkz/places365` → negative class (indoor/urban scenes — offices, streets, buildings)

**Output:** `nature_classifier.onnx` in `/kaggle/working/` — download this and commit to `ml/serve/model/`

**Expected runtime on T4 GPU: ~3–4 hours**

## Part 1 — Data Preparation

In [ ]:
!pip install -q --no-index --find-links=/kaggle/input/onnxruntime onnxruntime
!pip install -q --no-index --find-links=/kaggle/input/tf2onnx tf2onnx

import pathlib
import random
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Positive class: PlantNet-300K ---
# Directory is images_train (not images/train)
plantnet_base = pathlib.Path('/kaggle/input/plantnet-300k-images/plantnet_300K')
PLANT_DIR = plantnet_base / 'images_train'
pos_images = list(PLANT_DIR.rglob('*.jpg'))
print(f'Positive images found: {len(pos_images)}')

In [ ]:
# --- Negative class: Places365 ---
# 1.8M images across 365 indoor/urban scene categories — all clearly non-nature.
# A small set of nature-adjacent categories is excluded by name.

NATURE_ADJACENT = [
    'forest', 'park', 'garden', 'field', 'beach', 'mountain', 'lake',
    'river', 'waterfall', 'swamp', 'rainforest', 'marsh', 'tundra',
    'cliff', 'canyon', 'desert', 'campsite', 'farm', 'orchard',
    'pasture', 'lagoon', 'hayfield', 'rice_paddy', 'ice', 'snowfield',
    'ocean', 'coast', 'botanical', 'nature', 'outdoor'
]

# Auto-discover Places365 root — Kaggle mounts it under a 'places'-named subdir
neg_root = None
for candidate in pathlib.Path('/kaggle/input').iterdir():
    if 'places' in candidate.name.lower() and candidate.is_dir():
        neg_root = candidate
        break

neg_images = []
if neg_root:
    print(f'Found Places365 at: {neg_root}')
    for cat_dir in neg_root.rglob('*'):
        if not cat_dir.is_dir():
            continue
        cat_name = cat_dir.name.lower().replace('-', '_').replace(' ', '_')
        if any(kw in cat_name for kw in NATURE_ADJACENT):
            continue
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            neg_images.extend(cat_dir.glob(ext))
else:
    print("ERROR: Places365 not found. Add dataset via 'Add Data' → search 'places365'")

print(f'Negative images found: {len(neg_images)}')


In [ ]:
# --- Build balanced DataFrame and split ---
random.seed(42)
random.shuffle(neg_images)

sample_size = min(len(pos_images), len(neg_images))
assert sample_size > 0, (
    f"No images found! pos={len(pos_images)}, neg={len(neg_images)}. "
    "Make sure both datasets are added via 'Add Data' in the Kaggle notebook UI."
)

pos_rows = [{'path': str(p), 'label': 1} for p in pos_images[:sample_size]]
neg_rows = [{'path': str(p), 'label': 0} for p in neg_images[:sample_size]]

df = pd.DataFrame(pos_rows + neg_rows).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Total: {len(df)}  |  Positive: {df["label"].sum()}  |  Negative: {(df["label"]==0).sum()}')

train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=42)
val_df,   test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['label'], random_state=42)

train_df.to_csv('/kaggle/working/train.csv', index=False)
val_df.to_csv('/kaggle/working/val.csv',     index=False)
test_df.to_csv('/kaggle/working/test.csv',   index=False)
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}')

## Part 2 — EDA

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# Class balance
train_df['label'].value_counts().plot(kind='bar', color=['#ef4444','#22c55e'])
plt.xticks([0, 1], ['Non-nature (0)', 'Nature element (1)'], rotation=0)
plt.title('Class balance')
plt.tight_layout()
plt.show()

In [ ]:
# Sample positive images
pos_samples = train_df[train_df.label == 1].sample(25, random_state=42)['path'].tolist()
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for ax, path in zip(axes.flat, pos_samples):
    try:
        ax.imshow(Image.open(path).convert('RGB').resize((112, 112)))
    except Exception:
        ax.text(0.5, 0.5, 'Error', ha='center')
    ax.axis('off')
plt.suptitle('Positive samples (nature elements)')
plt.tight_layout()
plt.show()

## Part 3 — Training

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

IMG_SIZE   = 224
BATCH_SIZE = 32

train_df = pd.read_csv('/kaggle/working/train.csv')
val_df   = pd.read_csv('/kaggle/working/val.csv')

In [ ]:
def make_dataset(df, augment=False):
    paths  = df['path'].values
    labels = df['label'].values.astype('float32')

    def load_and_preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = preprocess_input(img)  # scales to [-1, 1] — must match app.py
        return img, label

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(buffer_size=10000, reshuffle_each_iteration=True)
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, augment=True)
val_ds   = make_dataset(val_df,   augment=False)

In [ ]:
# Build model — load weights by name with skip_mismatch (handles cross-version layer count diff)
import glob, pathlib

weights_candidates = glob.glob(
    '/kaggle/input/efficientnetb0b7-keras-weights/**/efficientnet-b0*notop*.h5',
    recursive=True
)
WEIGHTS_PATH = weights_candidates[0] if weights_candidates else None
print(f"Using weights: {WEIGHTS_PATH}")

# Build base without weights first, then load by name
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')
base   = EfficientNetB0(weights=None, include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Load weights by name — skips any mismatched layers from version differences
if WEIGHTS_PATH:
    base.load_weights(WEIGHTS_PATH, by_name=True, skip_mismatch=True)
    print("Weights loaded (by_name, skip_mismatch)")
else:
    print("WARNING: No weights file found — training from scratch (slower convergence)")

base.trainable = False

x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)
model.summary()

In [ ]:
# Phase 1 — train head only (base frozen)
model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
history1 = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)
print('Phase 1 complete')

In [ ]:
# Phase 2 — unfreeze last 30 layers for fine-tuning
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=10,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        ModelCheckpoint('/kaggle/working/nature_classifier.keras', save_best_only=True)
    ]
)
print('Phase 2 complete')

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
loss     = history1.history['loss']     + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
auc      = history1.history['auc']      + history2.history['auc']
val_auc  = history1.history['val_auc']  + history2.history['val_auc']
axes[0].plot(loss, label='Train'); axes[0].plot(val_loss, label='Val')
axes[0].axvline(len(history1.history['loss']), color='gray', linestyle='--')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(auc, label='Train'); axes[1].plot(val_auc, label='Val')
axes[1].axvline(len(history1.history['auc']), color='gray', linestyle='--')
axes[1].set_title('AUC'); axes[1].legend()
plt.tight_layout(); plt.show()

## Part 4 — Evaluation + ONNX Export

In [ ]:
# ── Patch 1: FuncGraph.__slots__ fix ────────────────────────────────────────
from tensorflow.python.framework.func_graph import FuncGraph

@property
def _captures_compat(self):
    c = None
    if hasattr(self, "captures"):
        c = self.captures
    elif hasattr(self, "function_captures"):
        c = self.function_captures
    if c is None:
        return {}
    if isinstance(c, dict):
        return c
    result = {}
    for t_val, t_name in c:
        try:
            result[t_val] = (t_val, t_name)
        except TypeError:
            pass
    return result

@_captures_compat.setter
def _captures_compat(self, value):
    pass

FuncGraph._captures = _captures_compat

# ── Patch 2: empty explicit_paddings fix ────────────────────────────────────
import onnx.helper as _onnx_helper

_orig_make_node = _onnx_helper.make_node

def _patched_make_node(op_type, inputs, outputs, name=None, doc_string=None, domain=None, **kwargs):
    clean = {k: v for k, v in kwargs.items()
             if not (isinstance(v, (list, tuple)) and len(v) == 0)}
    return _orig_make_node(op_type, inputs, outputs, name=name,
                           doc_string=doc_string, domain=domain, **clean)

_onnx_helper.make_node = _patched_make_node

# ── Patch 3: np.cast removed in NumPy 2.0 ───────────────────────────────────
import numpy as np
if not hasattr(np, 'cast'):
    class _NpCastAccessor:
        def __getitem__(self, dtype):
            return lambda x: np.array(x, dtype=dtype)
    np.cast = _NpCastAccessor()

import seaborn as sns
import onnxruntime as ort
import tf2onnx
from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix
from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

# ── ONNX Export via manual graph freezing ───────────────────────────────────
# from_keras can't freeze variables in TF 2.19 + NumPy 2.4 because tf2onnx's
# constant-fold rewriter is broken. Instead, we freeze with TF's own
# convert_variables_to_constants_v2 (pure C++, no NumPy), then hand the
# already-frozen GraphDef to from_graph_def — no variable capture needed.

@tf.function(input_signature=[tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name='input')])
def serving_fn(images):
    return model(images, training=False)

concrete_func = serving_fn.get_concrete_function()
frozen_func   = convert_variables_to_constants_v2(concrete_func)

input_names  = [t.name for t in frozen_func.inputs]
output_names = [t.name for t in frozen_func.outputs]
print(f'Frozen graph inputs:  {input_names}')
print(f'Frozen graph outputs: {output_names}')

model_proto, _ = tf2onnx.convert.from_graph_def(
    frozen_func.graph.as_graph_def(),
    input_names=input_names,
    output_names=output_names,
    opset=13
)

onnx_path = '/kaggle/working/nature_classifier.onnx'
with open(onnx_path, 'wb') as f:
    f.write(model_proto.SerializeToString())
print(f'ONNX model saved to {onnx_path}')

session    = ort.InferenceSession(onnx_path)
inputs_    = session.get_inputs()
print(f'ONNX input count: {len(inputs_)}')   # should be 1
input_name = inputs_[0].name
print(f'ONNX input tensor name: "{input_name}"')

In [ ]:
# Evaluate on test set
test_df = pd.read_csv('/kaggle/working/test.csv')

def predict(paths, batch_size=64):
    preds = []
    for i in range(0, len(paths), batch_size):
        batch = paths[i:i+batch_size]
        imgs = []
        for p in batch:
            try:
                img = tf.image.resize(
                    tf.image.decode_jpeg(tf.io.read_file(p), channels=3), [224, 224]
                ).numpy()
                imgs.append(preprocess_input(img))
            except Exception:
                imgs.append(np.zeros((224, 224, 3), dtype='float32'))
        arr = np.stack(imgs).astype('float32')
        preds.extend(session.run(None, {input_name: arr})[0].flatten().tolist())
    return np.array(preds)

y_pred = predict(test_df['path'].tolist())
y_true = test_df['label'].values

auc = roc_auc_score(y_true, y_pred)
print(f'AUC: {auc:.4f}  (target >= 0.90)')

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'AUC = {auc:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve'); plt.legend(); plt.tight_layout(); plt.show()

# Classification report at thresholds
for t in [0.85, 0.45]:
    print(f'\n=== Threshold = {t} ===')
    print(classification_report(y_true, (y_pred >= t).astype(int),
                                 target_names=['Non-nature', 'Nature element']))

# Confusion matrix
cm = confusion_matrix(y_true, (y_pred >= 0.85).astype(int))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-nature', 'Nature'], yticklabels=['Non-nature', 'Nature'])
plt.title('Confusion matrix (threshold=0.85)')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.tight_layout(); plt.show()

print('\nDone! Download nature_classifier.onnx from the output panel.')